# Importaçao das bibliotecas

In [13]:
import duckdb
import pathlib
import pandas as pd
from pathlib import Path
import pandas as pd
from pathlib import Path
from pyspark.sql.functions import input_file_name, split, col, lit, regexp_replace, element_at
from datetime import datetime, timedelta

# Caminho dos arquivos

In [36]:
fatura = Path(r'C:\Data_Lake_PoD_Cartoes\datalake\raw\fatura')
pagamentos = Path(r'C:\Data_Lake_PoD_Cartoes\datalake\raw\pagamentos')

# Criação das Vars de execução

In [37]:
ts_proc = datetime.now().strftime('%Y%m%d%H%M%S')
ts_proc

'20260728110506'

# Iniciar SparkSession

In [38]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Analise_DataLake").getOrCreate()

# Leitura Fatura

In [115]:
t_fatura = spark.read.option('header', 'true').csv(str(fatura), inferSchema=True)
t_fatura_00 = t_fatura.withColumn("filename", element_at(split(input_file_name(), r'[/\\]'), -1))
t_fatura_01 = t_fatura_00.withColumn('ref', split(col('filename'), '_')[2])
t_fatura_02 = t_fatura_01.withColumn('ts_file_generation', regexp_replace(split(col('filename'), '_')[3], '\.csv', ''))
t_fatura_03 = t_fatura_02.withColumn('ts_proc', lit(ts_proc))

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
C:\Users\gusau\AppData\Local\Temp\ipykernel_1196\317425876.py:4: SyntaxWarning: invalid escape sequence '\.'
  t_fatura_02 = t_fatura_01.withColumn('ts_file_generation', regexp_replace(split(col('filename'), '_')[3], '\.csv', ''))


In [116]:
t_fatura_03.createOrReplaceTempView("t_fatura_03")
t_fatura_03.cache()
t_fatura_03.count()


15553

In [117]:
t_fatura.printSchema()  

root
 |-- id_cliente: integer (nullable = true)
 |-- id_fatura: integer (nullable = true)
 |-- dt_emissao_fatura: date (nullable = true)
 |-- dt_vencimento_fatura: date (nullable = true)
 |-- valor_fatura: double (nullable = true)



In [118]:
t_fatura_04 = spark.sql("""
    select
        cast(replace(substr(dt_emissao_fatura,1,10),'-','') as string) ref,
        cast(ts_proc as string) as ts_proc,
        cast(ts_file_generation as string) ts_file_generation,
        cast(id_cliente as string) as id_cliente,
        cast(id_fatura as string) as id_fatura,
        cast(dt_emissao_fatura as timestamp) as dt_emissao_fatura,
        cast(dt_vencimento_fatura as timestamp) as dt_vencimento_fatura,
        cast(valor_fatura as decimal(14,2)) as valor_fatura
    from t_fatura_03
""")
t_fatura_04.createOrReplaceTempView("t_fatura_04")

In [119]:
t_fatura_04.show(2, truncate=False)

+--------+--------------+------------------+----------+---------+-------------------+--------------------+------------+
|ref     |ts_proc       |ts_file_generation|id_cliente|id_fatura|dt_emissao_fatura  |dt_vencimento_fatura|valor_fatura|
+--------+--------------+------------------+----------+---------+-------------------+--------------------+------------+
|20230101|20260728115902|20240902000000    |1         |1        |2023-01-01 00:00:00|2023-01-06 00:00:00 |1671.07     |
|20230201|20260728115902|20240902000000    |1         |2        |2023-02-01 00:00:00|2023-02-06 00:00:00 |3784.01     |
+--------+--------------+------------------+----------+---------+-------------------+--------------------+------------+
only showing top 2 rows


In [120]:
# Write

t_fatura_04.write.partitionBy('ref','ts_proc').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\fatura',mode='append')

In [124]:
t_fatura = spark.read.parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\fatura', inferSchema=True)

In [135]:
t_fatura.show(2, truncate=False)

+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
|ts_file_generation|id_cliente|id_fatura|dt_emissao_fatura  |dt_vencimento_fatura|valor_fatura|ref     |ts_proc       |
+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
|20240902000000    |1         |2        |2023-02-01 00:00:00|2023-02-06 00:00:00 |3784.01     |20230201|20260728115902|
|20240902000000    |2         |13       |2023-02-01 00:00:00|2023-02-06 00:00:00 |2806.11     |20230201|20260728115902|
+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
only showing top 2 rows


# Controle de Leitura

In [121]:
qtd_registros_fatura = t_fatura_04.count()

In [122]:
t_fatura_ctl = spark.sql(f"""
    select
        'fatura' as assunto,
        ref as ref,
        ts_file_generation as ts_file_generation,
        {ts_proc} as ts_proc,
        {qtd_registros_fatura} as qtd_registros
    from t_fatura_04
    limit 1
""")
t_fatura_ctl.createOrReplaceTempView('t_fatura_ctl')
t_fatura_ctl.show()

+-------+--------+------------------+--------------+-------------+
|assunto|     ref|ts_file_generation|       ts_proc|qtd_registros|
+-------+--------+------------------+--------------+-------------+
| fatura|20230101|    20240902000000|20260728115902|        15553|
+-------+--------+------------------+--------------+-------------+



In [123]:
# Write ctl file

t_fatura_ctl.write.partitionBy('ref').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\controle\controle_fatura',mode='append')

# Leitura Pagamentos

In [59]:
t_pagamentos = spark.read.csv(str(pagamentos), header=True, inferSchema=True)
t_pagamentos_00 = t_pagamentos.withColumn("filename", element_at(split(input_file_name(), r'[/\\]'), -1))
t_pagamentos_01 = t_pagamentos_00.withColumn('ref', split(col('filename'), '_')[2])
t_pagamentos_02 = t_pagamentos_01.withColumn('ts_file_generation', regexp_replace(split(col('filename'), '_')[3], '\.csv', ''))
t_pagamentos_03 = t_pagamentos_02.withColumn('ts_proc', lit(ts_proc))

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
C:\Users\gusau\AppData\Local\Temp\ipykernel_1196\370640367.py:4: SyntaxWarning: invalid escape sequence '\.'
  t_pagamentos_02 = t_pagamentos_01.withColumn('ts_file_generation', regexp_replace(split(col('filename'), '_')[3], '\.csv', ''))


In [60]:
t_pagamentos_03.createOrReplaceTempView('t_pagamentos_03')
t_pagamentos_03.cache()
t_pagamentos_03.count()

7709

In [62]:
t_pagamentos_03.printSchema() 

root
 |-- id_pagamento: integer (nullable = true)
 |-- id_fatura: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- valor_pagamento: double (nullable = true)
 |-- filename: string (nullable = true)
 |-- ref: string (nullable = true)
 |-- ts_file_generation: string (nullable = true)
 |-- ts_proc: string (nullable = false)



In [63]:
t_pagamentos_04 = spark.sql("""
    select
        cast(replace(substr(dt_pagamento,1,10),'-','') as string) as ref,
        cast(ts_file_generation as string) as ts_file_generation,
        cast(ts_proc as string) as ts_proc,
        cast(id_pagamento as string) as id_pagamento,
        cast(id_fatura as string) as id_fatura,
        cast(id_cliente as string) as id_cliente,
        cast(dt_pagamento as timestamp) as dt_pagamento,
        cast(valor_pagamento as decimal(14,2)) as valor_pagamento
    from t_pagamentos_03
""")
t_pagamentos_04.createOrReplaceTempView('t_pagamentos_04')

In [64]:
t_pagamentos_04.show(2, truncate=False)

+--------+------------------+--------------+------------+---------+----------+-------------------+---------------+
|ref     |ts_file_generation|ts_proc       |id_pagamento|id_fatura|id_cliente|dt_pagamento       |valor_pagamento|
+--------+------------------+--------------+------------+---------+----------+-------------------+---------------+
|20230211|20240902000000    |20260728110506|1           |2        |1         |2023-02-11 00:00:00|3784.01        |
|20230305|20240902000000    |20260728110506|2           |3        |1         |2023-03-05 00:00:00|3265.83        |
+--------+------------------+--------------+------------+---------+----------+-------------------+---------------+
only showing top 2 rows


In [65]:
#Write
t_pagamentos_04.write.partitionBy('ref','ts_proc').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\pagamentos',mode='append')

# Controle Pagamentos

In [66]:
qtd_registros_pagamentos = t_pagamentos_04.count()

In [67]:
t_pagamentos_ctl = spark.sql(f"""
    select
        'pagamentos' as assunto,
        ref as ref,
        ts_file_generation as ts_file_generation,
        {ts_proc} as ts_proc,
        {qtd_registros_pagamentos} as qtd_registros
    from t_pagamentos_04
    limit 1
""")
t_pagamentos_ctl.createOrReplaceTempView('t_pagamentos_ctl')

In [68]:
t_pagamentos_ctl.show()

+----------+--------+------------------+--------------+-------------+
|   assunto|     ref|ts_file_generation|       ts_proc|qtd_registros|
+----------+--------+------------------+--------------+-------------+
|pagamentos|20230211|    20240902000000|20260728110506|         7709|
+----------+--------+------------------+--------------+-------------+



In [74]:
#Write

t_pagamentos_ctl.write.partitionBy('ref','ts_proc').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\controle\controle_pagamentos',mode='append')

# Book de Variáveis

In [ ]:

# Volume de Transações
# Quantidade Total de Faturas
# Quantidade por Status (`PAGO_EM_DIA`, `PAGO_EM_ATRASO`, `PAGAMENTO_ANTECIPADO`, `NAO_PAGO`)
# Valores Financeiros:
# Valor Total de Faturas Emissões vs. Valor Total Pago
# Ticket Médio e Valores de Fatura por Status
# Indicadores de Risco:
# Máximo de Dias em Atraso por Cliente
# Janelas Temporais / Recorrência Histórica:
# Faixas Recentes (`1m`, `3m`, `6m`, `12m`)

In [ ]:
# Criação das Variáveis de Execução

In [98]:
dt_exec_book = datetime.now().date().strftime("%Y%m%d")
ts_proc = datetime.now().strftime('%Y%m%d%H%M%S')
ref_ini = (datetime.now().date() - timedelta(days=180)).strftime("%Y%m%d")
dt_exec_book, ref_ini

('20260728', '20260129')

#  Leitura Faturas

In [136]:
# Leitura Faturas
t_fatura = spark.read.parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\fatura', inferSchema=True)
t_fatura_00 = t_fatura.where(f'ref >= {ref_ini}')
t_fatura.createOrReplaceTempView('t_fatura_00')

In [137]:
t_fatura.count()

15553

# Deduplicação Faturas

In [ ]:
# Deduplicacao
t_fatura_dedup_00 = spark.sql("""
    select
        concat(ts_proc, ts_file_generation) as dedup_key,
        *
    from t_fatura_00
""")
t_fatura_dedup_00.createOrReplaceTempView("t_fatura_dedup_00")

In [ ]:
t_fatura_dedup_01 = spark.sql("""
    select
        id_cliente,
        id_fatura,
        max(dedup_key) as max_dedup_key
    from t_fatura_dedup_00
    group by id_cliente, id_fatura
""")
t_fatura_dedup_01.createOrReplaceTempView("t_fatura_dedup_01_keys")

+----------------------------+--------+
|dedup_key                   |count(1)|
+----------------------------+--------+
|2026072811590220240902000000|15553   |
+----------------------------+--------+



In [ ]:
t_fatura_dedup_final = spark.sql("""
    select
        a.*
    from t_fatura_dedup_00 a
    inner join t_fatura_dedup_01_keys b
        on  a.id_cliente = b.id_cliente
        and a.id_fatura  = b.id_fatura
        and a.dedup_key  = b.max_dedup_key
""")
t_fatura_dedup_final.createOrReplaceTempView("t_fatura_dedup_01")

In [140]:
t_fatura_dedup_01 = spark.sql("""
    select
        max(dedup_key) as max_dedup_key
    from t_fatura_dedup_00
""")
t_fatura_dedup_01.createOrReplaceTempView('t_fatura_dedup_01')

In [141]:
t_fatura_dedup_01 = spark.sql("""
    select
        a.*
    from t_fatura_dedup_00 a
    inner join t_fatura_dedup_01 b
    on a.dedup_key = b.max_dedup_key
""")
t_fatura_dedup_01.show(1)
t_fatura_dedup_01.createOrReplaceTempView('t_fatura_dedup_01')

+--------------------+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
|           dedup_key|ts_file_generation|id_cliente|id_fatura|  dt_emissao_fatura|dt_vencimento_fatura|valor_fatura|     ref|       ts_proc|
+--------------------+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
|20260728115902202...|    20240902000000|         1|        2|2023-02-01 00:00:00| 2023-02-06 00:00:00|     3784.01|20230201|20260728115902|
+--------------------+------------------+----------+---------+-------------------+--------------------+------------+--------+--------------+
only showing top 1 row


In [142]:
t_fatura_dedup_01.count()

15553

# Leitura Pagamentos

In [127]:
## Leitura Pagamentos
t_pagamentos = spark.read.parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\trusted\pagamentos')
t_pagamentos_00 = t_pagamentos.where(f'ref >= {ref_ini}')
t_pagamentos_00.createOrReplaceTempView('t_pagamentos_00')
t_pagamentos.createOrReplaceTempView('t_pagamentos')

In [128]:
t_pagamentos.count()

7709

# Deduplicação Pagamentos

In [129]:
t_pagamentos_dedup_00 = spark.sql("""
    select
        concat(ts_file_generation, ts_proc) dedup_key,
        *
    from t_pagamentos
""")
t_pagamentos_dedup_00.createOrReplaceTempView('t_pagamentos_dedup_00')

In [130]:
t_pagamentos_dedup_01 = spark.sql("""
    select
        max(dedup_key) max_dedup_key
    from t_pagamentos_dedup_00
""").createOrReplaceTempView('t_pagamentos_dedup_01')

In [131]:
t_pagamentos_dedup_02 = spark.sql("""
    select
        a.*
    from t_pagamentos_dedup_00 a
    inner join t_pagamentos_dedup_01 b
    on a.dedup_key = b.max_dedup_key
""")
t_pagamentos_dedup_02.createOrReplaceTempView('t_pagamentos_dedup_02')
t_pagamentos_dedup_02.count()

7709

In [132]:
t_pagamentos_dedup_02.show()

+--------------------+------------------+------------+---------+----------+-------------------+---------------+--------+--------------+
|           dedup_key|ts_file_generation|id_pagamento|id_fatura|id_cliente|       dt_pagamento|valor_pagamento|     ref|       ts_proc|
+--------------------+------------------+------------+---------+----------+-------------------+---------------+--------+--------------+
|20240902000000202...|    20240902000000|         182|      391|        26|2023-07-06 00:00:00|        4630.53|20230706|20260728110506|
|20240902000000202...|    20240902000000|         317|      721|        47|2023-07-06 00:00:00|         488.34|20230706|20260728110506|
|20240902000000202...|    20240902000000|         462|      997|        65|2023-07-06 00:00:00|        4620.56|20230706|20260728110506|
|20240902000000202...|    20240902000000|         721|     1517|        97|2023-07-06 00:00:00|        1948.41|20230706|20260728110506|
|20240902000000202...|    20240902000000|       

# Data Prep

In [143]:
t_fatura_pagamento_join = spark.sql("""
    select
        f.*,
        p.id_pagamento,
        p.dt_pagamento,
        p.valor_pagamento
    from t_fatura_dedup_01 f
    left join t_pagamentos_dedup_02 p
        on f.id_fatura = p.id_fatura 
       and f.id_cliente = p.id_cliente
""")
t_fatura_pagamento_join.createOrReplaceTempView('t_fatura_pagamento_join')

# Criação das Flags

In [ ]:
month_flags = spark.sql(f"""
    select
        id_cliente,
        id_fatura,
        id_pagamento,
        ref,
        dt_emissao_fatura,
        dt_vencimento_fatura,
        valor_fatura,
        dt_pagamento,
        valor_pagamento,

        (case
            when dt_pagamento is null then 'NAO_PAGO'
            when valor_pagamento < valor_fatura then 'PAGAMENTO_PARCIAL'
            when dt_pagamento < dt_vencimento_fatura then 'PAGAMENTO_ANTECIPADO'
            when dt_pagamento <= dt_vencimento_fatura then 'PAGO_EM_DIA'
            else 'PAGO_EM_ATRASO'
        end) as status_pagamento,

        (case
            when int(months_between(
                to_date('{dt_exec_book}','yyyyMMdd'),
                to_date(cast(ref as string),'yyyyMMdd')
            )) <= 1 then 1 else 0
        end) as flag_u1m,

        (case
            when int(months_between(
                to_date('{dt_exec_book}','yyyyMMdd'),
                to_date(cast(ref as string),'yyyyMMdd')
            )) <= 3 then 1 else 0
        end) as flag_u3m,

        (case
            when int(months_between(
                to_date('{dt_exec_book}','yyyyMMdd'),
                to_date(cast(ref as string),'yyyyMMdd')
            )) <= 6 then 1 else 0
        end) as flag_u6m,

        (case
            when int(months_between(
                to_date('{dt_exec_book}','yyyyMMdd'),
                to_date(cast(ref as string),'yyyyMMdd')
            )) <= 12 then 1 else 0
        end) as flag_u12m

    from t_fatura_pagamento_join
""")

month_flags.createOrReplaceTempView("month_flags")

# Criação do Book de Variáveis

In [158]:
bk_01 = spark.sql(f"""
    select
        id_cliente,
        '{dt_exec_book}' as ref,
        '{ts_proc}' as ts_proc,

        -- QTD por status
        sum(case when status_pagamento = 'PAGO_EM_DIA' then 1 end) as qtd_st_pago_em_dia,
        sum(case when status_pagamento = 'PAGO_EM_ATRASO' then 1 end) as qtd_st_pago_em_atraso,
        sum(case when status_pagamento = 'PAGAMENTO_ANTECIPADO' then 1 end) as qtd_st_pagamento_antecipado,
        sum(case when status_pagamento = 'NAO_PAGO' then 1 end) as qtd_st_nao_pago,
        sum(case when status_pagamento = 'PAGAMENTO_PARCIAL' then 1 end) as qtd_st_pagamento_parcial,

        -- QTD u1m
        sum(case when flag_u1m = 1 and status_pagamento = 'PAGO_EM_DIA' then 1 end) as qtd_st_pago_em_dia_u1m,
        sum(case when flag_u1m = 1 and status_pagamento = 'PAGO_EM_ATRASO' then 1 end) as qtd_st_pago_em_atraso_u1m,
        sum(case when flag_u1m = 1 and status_pagamento = 'PAGAMENTO_ANTECIPADO' then 1 end) as qtd_st_pagamento_antecipado_u1m,
        sum(case when flag_u1m = 1 and status_pagamento = 'NAO_PAGO' then 1 end) as qtd_st_nao_pago_u1m,
        sum(case when flag_u1m = 1 and status_pagamento = 'PAGAMENTO_PARCIAL' then 1 end) as qtd_st_pagamento_parcial_u1m,

        -- QTD u3m
        sum(case when flag_u3m = 1 and status_pagamento = 'PAGO_EM_DIA' then 1 end) as qtd_st_pago_em_dia_u3m,
        sum(case when flag_u3m = 1 and status_pagamento = 'PAGO_EM_ATRASO' then 1 end) as qtd_st_pago_em_atraso_u3m,
        sum(case when flag_u3m = 1 and status_pagamento = 'PAGAMENTO_ANTECIPADO' then 1 end) as qtd_st_pagamento_antecipado_u3m,
        sum(case when flag_u3m = 1 and status_pagamento = 'NAO_PAGO' then 1 end) as qtd_st_nao_pago_u3m,
        sum(case when flag_u3m = 1 and status_pagamento = 'PAGAMENTO_PARCIAL' then 1 end) as qtd_st_pagamento_parcial_u3m,

        -- QTD u6m
        sum(case when flag_u6m = 1 and status_pagamento = 'PAGO_EM_DIA' then 1 end) as qtd_st_pago_em_dia_u6m,
        sum(case when flag_u6m = 1 and status_pagamento = 'PAGO_EM_ATRASO' then 1 end) as qtd_st_pago_em_atraso_u6m,
        sum(case when flag_u6m = 1 and status_pagamento = 'PAGAMENTO_ANTECIPADO' then 1 end) as qtd_st_pagamento_antecipado_u6m,
        sum(case when flag_u6m = 1 and status_pagamento = 'NAO_PAGO' then 1 end) as qtd_st_nao_pago_u6m,
        sum(case when flag_u6m = 1 and status_pagamento = 'PAGAMENTO_PARCIAL' then 1 end) as qtd_st_pagamento_parcial_u6m,

        -- QTD u12m
        sum(case when flag_u12m = 1 and status_pagamento = 'PAGO_EM_DIA' then 1 end) as qtd_st_pago_em_dia_u12m,
        sum(case when flag_u12m = 1 and status_pagamento = 'PAGO_EM_ATRASO' then 1 end) as qtd_st_pago_em_atraso_u12m,
        sum(case when flag_u12m = 1 and status_pagamento = 'PAGAMENTO_ANTECIPADO' then 1 end) as qtd_st_pagamento_antecipado_u12m,
        sum(case when flag_u12m = 1 and status_pagamento = 'NAO_PAGO' then 1 end) as qtd_st_nao_pago_u12m,
        sum(case when flag_u12m = 1 and status_pagamento = 'PAGAMENTO_PARCIAL' then 1 end) as qtd_st_pagamento_parcial_u12m,

        -- SUM por status
        sum(case when status_pagamento = 'PAGO_EM_DIA' then valor_fatura end) as sum_st_pago_em_dia,
        sum(case when status_pagamento = 'PAGO_EM_ATRASO' then valor_fatura end) as sum_st_pago_em_atraso,
        sum(case when status_pagamento = 'PAGAMENTO_ANTECIPADO' then valor_fatura end) as sum_st_pagamento_antecipado,
        sum(case when status_pagamento = 'NAO_PAGO' then valor_fatura end) as sum_st_nao_pago,
        sum(case when status_pagamento = 'PAGAMENTO_PARCIAL' then valor_fatura end) as sum_st_pagamento_parcial


    from month_flags
    group by id_cliente
""")

bk_01.createOrReplaceTempView("bk_01")


In [ ]:
# write book_01
bk_01.write.partitionBy('ref','ts_proc').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\refined\book_01',mode='append')

# Controle Book

In [161]:
qtd_book = bk_01.count()
book_ctl = spark.sql(f"""
    select
        'Book_transacoes_01' as assunto,
        '{dt_exec_book}' as ref,
        '{ts_proc}' as ts_proc,
        {qtd_book} as qtd_registros
""")

In [164]:
book_ctl.show()

+------------------+--------+--------------+-------------+
|           assunto|     ref|       ts_proc|qtd_registros|
+------------------+--------+--------------+-------------+
|Book_transacoes_01|20260728|20260728115902|         1000|
+------------------+--------+--------------+-------------+



In [165]:
#write
book_ctl.write.partitionBy('ref','ts_proc').parquet(r'C:\Data_Lake_PoD_Cartoes\datalake\refined\controle_book_01',mode='append')